INITIALIZE

In [1]:
import pyvisa
import numpy as np
import time
import matplotlib.pyplot as plt
import os
import csv
%matplotlib qt

In [2]:
def update_plot(fig, ax, line, lstVoltage, lstAbsCurrent):
    line.set_data(lstVoltage, lstAbsCurrent)
    ax.relim()
    ax.autoscale_view()
    fig.canvas.draw()
    fig.canvas.flush_events()

In [3]:

rm = pyvisa.ResourceManager()

In [4]:
m2461 = rm.open_resource('TCPIP0::132.181.53.57::inst0::INSTR')
m2461.timeout = 10000
#m2461.write(f"reset()")

In [5]:
def fncMeasureOne(arrVMeasurement, \
                  device= 'pico',\
                  Sample = 'Test', sweepNum = 1,\
                  folderPath='230606\\Dev02\\Test',\
                  askToStart=False):
    if (device == 'pico'):
        fncMeasureOnePico(arrVMeasurement, Sample, sweepNum, folderPath, askToStart)
    elif (device == 'keith'):
        fncMeasureOneKL(arrVMeasurement, Sample, sweepNum, folderPath, askToStart)
    elif (device == 'keith1'):
        fncMeasureOneKL1(arrVMeasurement, Sample, sweepNum, folderPath, askToStart)
    elif (device == 'M2657A'):
        fncMeasureOneM2657A(arrVMeasurement, Sample, sweepNum, folderPath, askToStart)
    elif (device == 'M2461SWETSP'): #TSP means using TSP commands, SWE means sweep the whole thing from start to stop in one run
        fncMeasureOneM2461SWETSP(arrVMeasurement, Sample, sweepNum, folderPath, askToStart)
    elif (device == 'M2461PULSESWETSP'): #TSP means using TSP commands, SWE means sweep the whole thing from start to stop in one run
        fncMeasureOneM2461PULSESWETSP(arrVMeasurement, Sample, sweepNum, folderPath, askToStart)
    elif (device == 'M2461POITSP'): #POI means set each point and then measure and move to another.
        fncMeasureOneM2461POITSP(arrVMeasurement, Sample, sweepNum, folderPath, askToStart)
    elif (device == 'M2461SWE'): #No TSP means using SCPI commands, SWE means sweep the whole thing from start to stop in one run
        fncMeasureOneM2461SWE(arrVMeasurement, Sample, sweepNum, folderPath, askToStart)
    else:
        print("Incorrect device name")

In [6]:
def fncMeasureMultiple(settings =['pico', [[0, 0.5, 0.1]]], \
                  Sample = 'Test',\
                  startNum = 1, \
                  folderPath='230606\\Dev02\\Test',\
                  askToStart=False): 
    sweepNum = 0
    dev, arrVMeasurements = settings
    for j in range (len(arrVMeasurements)):
        sweepNum = j + startNum
        fncMeasureOne(arrVMeasurements[j], dev , Sample, sweepNum, folderPath)
        time.sleep(5)

In [7]:
def setM2461IVMeasurementTSP (terminal = "FRONT",\
                  vRange = "AUTO",\
                  iRange = "AUTO",\
                  iCompliance = 7,\
                  iMeasMode = "2WIRE",\
                  iNPLC = 1,\
                  delay = 0.1): #(iRange = "AUTO", iLimit = 7, nplc = 1, delay = 0.1)
    # Reset and initialize instrutment
    m2461.write(f"defbuffer1.clear()")
    m2461.write(f"buffer.clearstats()")
    
    m2461.write(f"smu.reset()")
    m2461.write(f"status.reset()")
    m2461.write(f"errorqueue.clear()")
    m2461.write(f"dataqueue.clear()")
    
    
    ##############################################
    ## Make sure that no data left in the buffer
    # Send command to check the number of stored readings in defbuffer1
    m2461.write("print(defbuffer1.n)")
    # Read the response and convert it to an integer
    buffer_count = round(float(m2461.read().strip()))
    print(f"Number of data points in buffer: {buffer_count}")
    if buffer_count != 0:
        data = m2461.read()
        print(f"data remains in buffer: {data}")
    
    ##############################################
    m2461.write(f"smu.terminals = smu.TERMINALS_{terminal}" )

    # Setup current measurement mode
    m2461.write(f"smu.measure.sense = smu.SENSE_{iMeasMode}")

     # Setup voltage source mode
    m2461.write(f"smu.source.func = smu.FUNC_DC_VOLTAGE")

    if vRange == "AUTO":
        m2461.write(f"smu.source.autorange = smu.ON")  
    else:
        m2461.write(f"smu.source.autorange = smu.OFF")
        m2461.write(f"smu.source.range = {vRange}") 
    
    #Set up current compliance
    m2461.write(f"smu.source.ilimit.level = %f" % iCompliance)

    #Set up current measurement range
    if iRange == "AUTO":
        m2461.write(f"smu.measure.autorange = smu.ON")
    else:
        m2461.write(f"smu.measure.autorange = smu.OFF")
        m2461.write(f"smu.measure.range = %f" % iRange)

   
    
                    
    

    #Set the measurement integration time and delay
    m2461.write(f"smu.measure.nplc = %f" % iNPLC)
    m2461.write(f"smu.source.delay = %f" % delay)

    m2461.write(f"beeper.beep(0.5,100)")
    return True
setM2461IVMeasurementTSP()   
  
    
    #m2461.write(f"display.changescreen(display.SCREEN_HOME)")

def measureM2461OnePointTSP(setV, outputON = False, resetV = True):
    #setM2461IVMeasurement()
    #m2461.write(f"bufData = buffer.make(1)") 
    m2461.write(f"smu.source.func = smu.FUNC_DC_VOLTAGE")
    m2461.write(f"smu.source.level = %f" %setV)
    
    if (not outputON):
        m2461.write(f"smu.source.output = 1")
    
    m2461.write(f"smu.measure.func = smu.FUNC_DC_VOLTAGE")
    m2461.write(f"print(smu.measure.read())") 
    voltage = m2461.read()

    m2461.write(f"smu.measure.func = smu.FUNC_DC_CURRENT")
    m2461.write(f"print(smu.measure.read())") 
    #m2461.write(f'printbuffer(1, 1, defbuffer1.sourcevalues, defbuffer1.readings)')
    current = m2461.read()
    #m2461.write(f"buffer.delete(bufData)")

    #Turn off the output
    if resetV:
        m2461.write(f"smu.source.level = 0")
        m2461.write(f"smu.source.output = 0")
    return [float(current), float(voltage)] #[float(arrRawData[i]) for i in range(len(arrRawData))]

# rawData
current, voltage = measureM2461OnePointTSP(0.1,False,True)
print ("Current: ", current, "Voltage: ", voltage)
#print (rawData)


Number of data points in buffer: 0
Current:  1.7801912722e-11 Voltage:  0.10000796616


In [8]:
def fncMeasureOneM2461POITSP(arrVMeasurement, \
                  Sample = 'Test', sweepNum = 1,\
                  folderPath='250319\\Dev02\\Test',\
                  askToStart=False,\
                  terminal = "FRONT",\
                  vRange = "AUTO",\
                  iRange = "AUTO",\
                  iCompliance = 7,\
                  iMeasMode = "2WIRE", \
                  iNPLC = 1, \
                  delay = 0.1,\
                  ):
    Vstart, Vstop, Vstep = arrVMeasurement
    fileName = "Sweep"+str(sweepNum)+Sample+'Star'+str(Vstart).replace("-","M")+'_Stop'+str(Vstop).replace("-","M")+'_Step'+str(Vstep).replace("-", "M")+'.csv'
    filePath = os.path.join(folderPath, fileName)
    print(filePath)
    stepNum = np.floor(abs((Vstart - Vstop)/Vstep)).astype(int) + 1

    setM2461IVMeasurementTSP(terminal, vRange, iRange, iCompliance, iMeasMode, iNPLC, delay)
    absVstart = abs(Vstart)
    absVstop = abs(Vstop)
    # Set up source range
    if abs(Vstart) > abs(Vstop): 
        m2461.write(f"smu.source.rangev = %f" % absVstart)
    else:
        m2461.write(f"smu.source.rangev = %f" % absVstop)
    
    lstCurrent = []
    lstVoltage = []
    lstAbsCurrent = []
    
    fig, ax = plt.subplots()
    line, = ax.plot([], [], 'o-')
    ax.set_xlabel('Voltage (V)')
    ax.set_ylabel('Absolute Current (A)')
    ax.set_yscale('log')
    plt.show()
    time.sleep(1)
    m2461.write(f"smu.source.output = 1")
   
    # Measurement loop
    for i in range(stepNum):
        current, voltage = measureM2461OnePointTSP(Vstart + i*Vstep,True,False)
        lstVoltage.append(voltage)
        lstCurrent.append(current)
        lstAbsCurrent.append(abs(current))
        update_plot(fig, ax, line,lstVoltage, lstAbsCurrent)
        #time.sleep(0.1)
    #Turn off the output
    m2461.write(f"smu.source.level = 0")
    m2461.write(f"smu.source.output = 0")
    dataToWrite = [['vRange: ', vRange]]
    dataToWrite.append(['iLimit: ', iCompliance])
    dataToWrite.append(['iRange: ', iRange[16:]])
    dataToWrite.append(['nplc: ', iNPLC, 'delay: ', delay])
    dataToWrite.append(['data Number:', stepNum])
    dataToWrite.append(['V_meas', 'I_meas'])

    for i in range(len(lstCurrent)):
        dataToWrite.append([lstVoltage[i], lstCurrent[i]])
    try:
        with open(filePath, 'w', newline = '') as file:
            writer = csv.writer(file)
            writer.writerows(dataToWrite)
            file.close()
    except IOError:
        print("Error: Could not open file for writing")
    plt.show()
    plt.savefig(os.path.splitext(filePath)[0] + '_log.png')
    ax.set_yscale('linear')
    plt.show()
    plt.savefig(os.path.splitext(filePath)[0] + '_lin.png')

In [9]:
def fncMeasureOneM2461SWETSP(arrVMeasurement, \
                  Sample = 'Test', sweepNum = 1,\
                  folderPath='250318\\Dev02\\Test',\
                  askToStart=False,\
                  terminal = "FRONT",\
                  vRange = "AUTO",\
                  iRange = "AUTO",\
                  iCompliance = 7,\
                  iMeasMode = "2WIRE",\
                  iNPLC = 1,\
                  delay = 0.1
                  ):
    lstCurrent = []
    lstVoltage = []
    lstAbsCurrent = []
    lstCalVoltage = []

    Vstart, Vstop, Vstep = arrVMeasurement
    fileName = "Sweep"+str(sweepNum)+Sample+'Star'+str(Vstart).replace("-","M")+'_Stop'+str(Vstop).replace("-","M")+'_Step'+str(Vstep).replace("-", "M")+'.csv'
    filePath = os.path.join(folderPath, fileName)
    print(filePath)
    stepNum = np.floor(abs((Vstart - Vstop)/Vstep)).astype(int) + 1
    
    setM2461IVMeasurementTSP(terminal, vRange, iRange, iCompliance, iMeasMode, iNPLC, delay)
    
    m2461.write(f'smu.source.sweeplinear("VLinSweep", {Vstart:0.2f}, {Vstop:0.2f}, {stepNum:f}, {delay:0.2f})')
        
    m2461.write(f'trigger.model.initiate()')
    m2461.write(f'waitcomplete()')
    
    
    m2461.write(f'printbuffer(1, {stepNum:f}, defbuffer1.sourcevalues, defbuffer1.readings)')
    m2461.write(f'waitcomplete()')
    ## ensure that the measurement completes before retrieving data
    time.sleep((delay+0.15)*stepNum) # 
    ## ensure that the measurement completes before retrieving data
    data = m2461.read().split(', ') # this happens ansychonously with the sweep.
    print('Measured data: ', data)
    lstCurrent = [float(i) for i in data[1::2]] # get all odd elements, which are currents

    lstVoltage = [float(i) for i in data[0::2]] # get all even elements, which are voltages
    print (f'len current: {len(lstCurrent)}')
    print (f'len current: {len(lstVoltage)}')
    if Vstart > Vstop:
        Vstep = -abs(Vstep)
    else:
        Vstep = abs(Vstep)
    

    lstAbsCurrent = np.abs(lstCurrent)
    lstCalVoltage = [(Vstart + i*Vstep) for i in range(stepNum)]
    dataToWrite = [['vRange', vRange]]
    dataToWrite.append(['iCompliance', iCompliance])
    #dataToWrite.append(['iRange', iRange[16:]])
    dataToWrite.append(['iNPLC', iNPLC])
    dataToWrite.append(['data Number', stepNum])
    dataToWrite = [['V', 'I', 'V_cal','abs_I']]
    for i in range(len(lstCurrent)):
        dataToWrite.append([lstVoltage[i], lstCurrent[i], lstCalVoltage[i] , lstAbsCurrent[i]])
    try:
        with open(filePath, 'w', newline = '') as file:
            writer = csv.writer(file)
            writer.writerows(dataToWrite)
            file.close()
    except IOError:
        print("Error: Could not open file for writing")

    fig, ax = plt.subplots()
    line, = ax.plot(lstVoltage, lstAbsCurrent, 'o-')
    ax.set_xlabel('Voltage (V)')
    ax.set_ylabel('Current (A)')
    ax.set_yscale('linear')
    plt.show()
    plt.savefig(os.path.splitext(filePath)[0] + '_lin.png')
    ax.set_yscale('log')
    plt.show()
    plt.savefig(os.path.splitext(filePath)[0] + '_log.png')   


In [10]:
def fncMeasureOneM2461PULSESWETSP(arrVMeasurement, \
                  Sample = 'Test', sweepNum = 1,\
                  folderPath='250318\\Dev02\\Test',\
                  askToStart=False,\
                  terminal = "FRONT",\
                  vRange = 7,\
                  iRange = 10,\
                  iCompliance = 7,\
                  iMeasMode = "2WIRE",\
                  iNPLC = 0.01,\
                  delay = 0\
                  ):
    lstCurrent = []
    lstVoltage = []
    lstAbsCurrent = []
    lstCalVoltage = []

    Vstart, Vstop, Vstep, pulseWidth, offTime = arrVMeasurement
    fileName = "Sweep"+str(sweepNum)+"Pulse"+Sample+'Star'+str(Vstart).replace("-","M").replace(".","p")+\
        '_Stop'+str(Vstop).replace("-","M").replace(".","p")+'_Step'+str(Vstep).replace("-", "M").replace(".","p")+\
        'pWidth'+str(pulseWidth).replace(".","p")+'offTime'+str(offTime).replace(".","p")+\
        '.csv'
    filePath = os.path.join(folderPath, fileName)
    print(filePath)
    stepNum = np.floor(abs((Vstart - Vstop)/Vstep)).astype(int) + 1
    setM2461IVMeasurementTSP(terminal, vRange, iRange, iCompliance, iMeasMode, iNPLC, delay)
    
    m2461.write(f'smu.source.readback = smu.OFF') 
    m2461.write(f'smu.measure.autozero.enable = smu.OFF')

    # m2461.write(f'smu.measure.autozero.enable = smu.OFF')

    # m2461.write(f'smu.source.pulsesweeplinear("VoltPulseLinSweepList",0 , {Vstart:0.2f}, {Vstop:0.2f}, {stepNum:f}, {pulseWidth:0.6f},\
    #             smu.ON, defbuffer1, {delay:0.6f}, {offTime:0.6f}, 1, 7, 10, smu.OFF, smu.OFF') 
    m2461.write(f'smu.source.pulsesweeplinear("VoltPulseLinSweepList", 0, {Vstart:0.2f}, {Vstop:0.2f}, {stepNum:f}, {pulseWidth:0.6f}, \
                smu.ON, defbuffer1, {delay:0.6f}, {offTime:0.6f}, 1, {vRange:0.2f}, {iRange:0.2f}, smu.OFF, smu.OFF)')    
    m2461.write(f'trigger.model.initiate()')
    m2461.write(f'waitcomplete()')
    
    
    m2461.write(f'printbuffer(1, {stepNum:f}, defbuffer1.sourcevalues, defbuffer1.readings)')
    m2461.write(f'waitcomplete()')
    ## ensure that the measurement completes before retrieving data
    time.sleep((delay+0.25)*stepNum) # 
    ## ensure that the measurement completes before retrieving data
    data = m2461.read().split(', ') # this happens ansychonously with the sweep.
    print('Measured data: ', data)
    lstCurrent = [float(i) for i in data[1::2]] # get all odd elements, which are currents

    lstVoltage = [float(i) for i in data[0::2]] # get all even elements, which are voltages
    print (f'len current: {len(lstCurrent)}')
    print (f'len current: {len(lstVoltage)}')
    if Vstart > Vstop:
        Vstep = -abs(Vstep)
    else:
        Vstep = abs(Vstep)
    

    lstAbsCurrent = np.abs(lstCurrent)
    lstCalVoltage = [(Vstart + i*Vstep) for i in range(stepNum)]
    dataToWrite = [['vRange', vRange]]
    dataToWrite.append(['iCompliance', iCompliance])
    #dataToWrite.append(['iRange', iRange[16:]])
    dataToWrite.append(['iNPLC', iNPLC])
    dataToWrite.append(['data Number', stepNum])
    dataToWrite = [['V', 'I', 'V_cal','abs_I']]
    for i in range(len(lstCurrent)):
        dataToWrite.append([lstVoltage[i], lstCurrent[i], lstCalVoltage[i] , lstAbsCurrent[i]])
    try:
        with open(filePath, 'w', newline = '') as file:
            writer = csv.writer(file)
            writer.writerows(dataToWrite)
            file.close()
    except IOError:
        print("Error: Could not open file for writing")

    fig, ax = plt.subplots()
    line, = ax.plot(lstVoltage, lstAbsCurrent, 'o-')
    ax.set_xlabel('Voltage (V)')
    ax.set_ylabel('Current (A)')
    ax.set_yscale('linear')
    plt.show()
    plt.savefig(os.path.splitext(filePath)[0] + '_lin.png')
    ax.set_yscale('log')
    plt.show()
    plt.savefig(os.path.splitext(filePath)[0] + '_log.png')  

In [11]:
def fncMeasureOneM2461SWE(arrVMeasurement, \
                  Sample = 'Test', sweepNum = 1,\
                  folderPath='250318\\Dev02\\Test',\
                  askToStart=False):
    Vstart, Vstop, Vstep = arrVMeasurement
    fileName = "Sweep"+str(sweepNum)+Sample+'Star'+str(Vstart).replace("-","M")+'_Stop'+str(Vstop).replace("-","M")+'_Step'+str(Vstep).replace("-", "M")+'.csv'
    filePath = os.path.join(folderPath, fileName)
    print(filePath)
    stepNum = np.floor(abs((Vstart - Vstop)/Vstep)).astype(int) + 1
    vRange = "AUTO"
    iCompliance = 7
    iNPLC = 0.01
    delay = 0.0
    # data = m2461.query_ascii_values('FETC?')
    # print('this is the data 1', data)
    m2461.write('*RST')
    m2461.write("TRAC:CLE")
    m2461.write(':ROUT:TERM FRON')
    m2461.write(':SOUR:FUNC VOLT')
    # vRange = ':SOUR:VOLT:RANG:' + vRange
    # print(vRange)
    m2461.write(':SOUR:VOLT:RANG:AUTO ON')
    m2461.write('SOUR:VOLT:ILIM %0.2f' % iCompliance)
    m2461.write(':SENS:FUNC "CURR"')
    m2461.write('SENS:CURR:RANG:AUTO ON')
    m2461.write(':SENSE:CURR:NPLC %f' % iNPLC )
    m2461.write('SOUR:SWE:VOLT:LIN %0.2f, %0.2f, %f, %0.2f' % (Vstart, Vstop, stepNum, delay))
    m2461.write('INIT')
    #m2461.write('*WAI')
    m2461.write('*OPC')
    m2461.write('TRAC:DATA? 1, %f, "defbuffer1", SOUR, READ' % stepNum)
    #m2461.write('*WAI')
    m2461.write('*OPC')
    data = m2461.read().strip().split(',')
    
    #data = m2461.query_ascii_values('FETC?')
    
    print('Measured data: ', data)
   
    lstCurrent = [ float(i) for i in data[1::2]]# get all odd elements, which are currents

    lstVoltage = [float(v) for v in data[0::2]] # get all even elements, which are voltages

    # if Vstart > Vstop:
    #     Vstep = -abs(Vstep)
    # else:
    #     Vstep = abs(Vstep)
    lstAbsCurrent = [abs(i) for i in lstCurrent]
    lstCalVoltage = [(Vstart + i*Vstep) for i in range(stepNum)]
    dataToWrite = [['vRange', vRange]]
    dataToWrite.append(['iCompliance', iCompliance])
    #dataToWrite.append(['iRange', iRange[16:]])
    dataToWrite.append(['iNPLC', iNPLC])
    dataToWrite.append(['data Number', stepNum])
    dataToWrite = [['V', 'I', 'V_cal','abs_I']]
    for i in range(len(lstCurrent)):
        dataToWrite.append([lstVoltage[i], lstCurrent[i], lstCalVoltage[i] , lstAbsCurrent[i]])
    try:
        with open(filePath, 'w', newline = '') as file:
            writer = csv.writer(file)
            writer.writerows(dataToWrite)
            file.close()
    except IOError:
        print("Error: Could not open file for writing")

    fig, ax = plt.subplots()
    line, = ax.plot(lstVoltage, lstAbsCurrent, 'o-')
    ax.set_xlabel('Voltage (V)')
    ax.set_ylabel('Current (A)')
    ax.set_yscale('linear')
    plt.show()
    plt.savefig(os.path.splitext(filePath)[0] + '_lin.png')
    ax.set_yscale('log')
    plt.show()
    plt.savefig(os.path.splitext(filePath)[0] + '_log.png')
    
    
    
    

In [ ]:
folderPath = '../Devices/250828_Fab250828_O6887_NE_RecessGateNICP'
# Create a new directory
try:
    os.mkdir(folderPath)
except FileExistsError:
    print("Folder already exists")
    
folderPath = folderPath+ '\\Dev07'
# Create a new directory
try:
    os.mkdir(folderPath)
except FileExistsError:
    print("Folder already exists")

folderPath = folderPath+ '\\M01'
# Create a new directory
try:
    os.mkdir(folderPath)
except FileExistsError:
    print("Folder already exists")
sample = folderPath[-3:]

Folder already exists
Folder already exists
Folder already exists


In [20]:
#run Keithley 2400 
arrKLVMeasMulti =["M2461PULSESWETSP",[[0, 3.0,0.01,1e-3,0.1],\
                            #[0, 3, 0.01],\
                            [0, -3.0, 0.01,1e-3,0.1]
                            ]]
fncMeasureMultiple(arrKLVMeasMulti,sample,17, folderPath)

../Devices/250813_Fab250812_O6687NE_RecessGate\Dev13\M01\Sweep17PulseM01Star0_Stop3p0_Step0p01pWidth0p001offTime0p1.csv
Number of data points in buffer: 0
Measured data:  ['0', '1.1444091797e-05', '0.01', '0.00012588500977', '0.02', '-0.00010681152344', '0.03', '1.1444091797e-05', '0.04', '-0.00010681152344', '0.05', '0.00012588500977', '0.06', '1.1444091797e-05', '0.07', '1.1444091797e-05', '0.08', '-0.00010681152344', '0.09', '-0.00022125244141', '0.1', '-0.00010681152344', '0.11', '1.1444091797e-05', '0.12', '-0.00010681152344', '0.13', '1.1444091797e-05', '0.14', '-0.00010681152344', '0.15', '1.1444091797e-05', '0.16', '1.1444091797e-05', '0.17', '-0.00010681152344', '0.18', '-0.00010681152344', '0.19', '1.1444091797e-05', '0.2', '1.1444091797e-05', '0.21', '1.1444091797e-05', '0.22', '1.1444091797e-05', '0.23', '1.1444091797e-05', '0.24', '-0.00010681152344', '0.25', '-0.00022125244141', '0.26', '-0.00010681152344', '0.27', '-0.00010681152344', '0.28', '-0.00010681152344', '0.29',

In [ ]:
#run Keithley 2400 
arrKLVMeasMulti =["M2461SWETSP",[[0, 3,0.05],\
                            #[0, 3, 0.01],\
                            [0, -3, 0.05]
                            ]]
fncMeasureMultiple(arrKLVMeasMulti,sample, 13, folderPath)

In [ ]:

m2461.write('TRAC:DATA? 1, %f, "defbuffer1", SOUR, READ' % stepNum)
    #m2461.write('*WAI')
m2461.write('*OPC')
data = m2461.read().strip().split(',')

['0.0000058059959', '-4.817887061526e-11', '0.0500051006675', '-1.604667093646e-11', '0.1000043898821', '-4.588372043424e-11', '0.1500057578087', '-4.358857025322e-11', '0.2000088691711', '-2.235831658703e-11', '0.2500146329403', '-3.498162176596e-11', '0.3000109493732', '-2.522721614939e-11', '0.3500224649906', '-3.67029531767e-11', '0.4000218212605', '-2.522717451603e-11', '0.4500183463097', '-2.121069292427e-11', '0.5000233054161', '-3.268642995158e-11', '0.5500213503838', '-1.489896400697e-11', '0.6000233292580', '-1.203006444461e-11', '0.6500200629234', '5.933240010414e-13', '0.7000154256821', '-3.996976360998e-12', '0.7500165700912', '4.190611052612e-11', '0.8000249862671', '1.28548102718e-10', '0.8500210642815', '6.317578682413e-10', '0.9000239372253', '3.167887507871e-09', '0.9500221610069', '1.577222796811e-08', '1.0000206232071', '7.192454631877e-08', '1.0500218868256', '2.852647469354e-07', '1.1000128984451', '8.358048830814e-07', '1.1500061750412', '2.376675638516e-06', '1.

In [41]:
#in case probram crashed because of hitting limit
#data = ['0', '-0.00010681152344', '-0.01', '-0.00010681152344', '-0.02', '-0.00010681152344', '-0.03', '1.1444091797e-05', '-0.04', '1.1444091797e-05', '-0.05', '-0.00022125244141', '-0.06', '-0.00010681152344', '-0.07', '-0.00022125244141', '-0.08', '1.1444091797e-05', '-0.09', '-0.00010681152344', '-0.1', '1.1444091797e-05', '-0.11', '1.1444091797e-05', '-0.12', '-0.00010681152344', '-0.13', '-0.00010681152344', '-0.14', '1.1444091797e-05', '-0.15', '1.1444091797e-05', '-0.16', '-0.00010681152344', '-0.17', '-0.00022125244141', '-0.18', '-0.00010681152344', '-0.19', '-0.00010681152344', '-0.2', '-0.00010681152344', '-0.21', '1.1444091797e-05', '-0.22', '-0.00010681152344', '-0.23', '-0.00010681152344', '-0.24', '-0.00010681152344', '-0.25', '-0.00010681152344', '-0.26', '1.1444091797e-05', '-0.27', '-0.00022125244141', '-0.28', '-0.00022125244141', '-0.29', '-0.00010681152344', '-0.3', '1.1444091797e-05', '-0.31', '-0.00010681152344', '-0.32', '1.1444091797e-05', '-0.33', '-0.00022125244141', '-0.34', '1.1444091797e-05', '-0.35', '-0.00010681152344', '-0.36', '1.1444091797e-05', '-0.37', '-0.00033569335938', '-0.38', '-0.00010681152344', '-0.39', '-0.00010681152344', '-0.4', '-0.00010681152344', '-0.41', '-0.00010681152344', '-0.42', '-0.00022125244141', '-0.43', '-0.00010681152344', '-0.44', '1.1444091797e-05', '-0.45', '-0.00010681152344', '-0.46', '-0.00010681152344', '-0.47', '1.1444091797e-05', '-0.48', '-0.00033569335938', '-0.49', '-0.00022125244141', '-0.5', '1.1444091797e-05', '-0.51', '-0.00010681152344', '-0.52', '-0.00010681152344', '-0.53', '1.1444091797e-05', '-0.54', '-0.00022125244141', '-0.55', '-0.00010681152344', '-0.56', '1.1444091797e-05', '-0.57', '-0.00010681152344', '-0.58', '0.00012588500977', '-0.59', '-0.00010681152344', '-0.6', '-0.00010681152344', '-0.61', '-0.00010681152344', '-0.62', '1.1444091797e-05', '-0.63', '1.1444091797e-05', '-0.64', '-0.00010681152344', '-0.65', '-0.00010681152344', '-0.66', '-0.00010681152344', '-0.67', '-0.00022125244141', '-0.68', '-0.00010681152344', '-0.69', '-0.00010681152344', '-0.7', '-0.00010681152344', '-0.71', '0.00012588500977', '-0.72', '-0.00022125244141', '-0.73', '-0.00010681152344', '-0.74', '1.1444091797e-05', '-0.75', '-0.00010681152344', '-0.76', '-0.00022125244141', '-0.77', '-0.00010681152344', '-0.78', '1.1444091797e-05', '-0.79', '1.1444091797e-05', '-0.8', '1.1444091797e-05', '-0.81', '-0.00010681152344', '-0.82', '1.1444091797e-05', '-0.83', '1.1444091797e-05', '-0.84', '-0.00010681152344', '-0.85', '0.00012588500977', '-0.86', '-0.00010681152344', '-0.87', '1.1444091797e-05', '-0.88', '1.1444091797e-05', '-0.89', '-0.00010681152344', '-0.9', '-0.00010681152344', '-0.91', '-0.00010681152344', '-0.92', '1.1444091797e-05', '-0.93', '1.1444091797e-05', '-0.94', '-0.00010681152344', '-0.95', '-0.00022125244141', '-0.96', '-0.00022125244141', '-0.97', '1.1444091797e-05', '-0.98', '1.1444091797e-05', '-0.99', '-0.00010681152344', '-1', '1.1444091797e-05', '-1.01', '-0.00010681152344', '-1.02', '0.00012588500977', '-1.03', '-0.00010681152344', '-1.04', '-0.00010681152344', '-1.05', '1.1444091797e-05', '-1.06', '-0.00022125244141', '-1.07', '-0.00010681152344', '-1.08', '1.1444091797e-05', '-1.09', '-0.00010681152344', '-1.1', '-0.00010681152344', '-1.11', '1.1444091797e-05', '-1.12', '-0.00010681152344', '-1.13', '-0.00010681152344', '-1.14', '-0.00022125244141', '-1.15', '-0.00010681152344', '-1.16', '-0.00010681152344', '-1.17', '-0.00010681152344', '-1.18', '1.1444091797e-05', '-1.19', '1.1444091797e-05', '-1.2', '-0.00010681152344', '-1.21', '0.00012588500977', '-1.22', '1.1444091797e-05', '-1.23', '-0.00010681152344', '-1.24', '-0.00010681152344', '-1.25', '1.1444091797e-05', '-1.26', '0.00012588500977', '-1.27', '-0.00010681152344', '-1.28', '-0.00010681152344', '-1.29', '1.1444091797e-05', '-1.3', '0.00012588500977', '-1.31', '1.1444091797e-05', '-1.32', '1.1444091797e-05', '-1.33', '1.1444091797e-05', '-1.34', '-0.00010681152344', '-1.35', '1.1444091797e-05', '-1.36', '-0.00010681152344', '-1.37', '-0.00010681152344', '-1.38', '-0.00010681152344', '-1.39', '-0.00010681152344', '-1.4', '-0.00022125244141', '-1.41', '0.00012588500977', '-1.42', '-0.00010681152344', '-1.43', '-0.00010681152344', '-1.44', '-0.00010681152344', '-1.45', '0.00024032592773', '-1.46', '-0.00010681152344', '-1.47', '1.1444091797e-05', '-1.48', '-0.00010681152344', '-1.49', '1.1444091797e-05', '-1.5', '-0.00010681152344', '-1.51', '-0.00010681152344', '-1.52', '1.1444091797e-05', '-1.53', '-0.00010681152344', '-1.54', '1.1444091797e-05', '-1.55', '-0.00010681152344', '-1.56', '1.1444091797e-05', '-1.57', '-0.00010681152344', '-1.58', '-0.00022125244141', '-1.59', '1.1444091797e-05', '-1.6', '1.1444091797e-05', '-1.61', '-0.00010681152344', '-1.62', '1.1444091797e-05', '-1.63', '-0.00022125244141', '-1.64', '-0.00010681152344', '-1.65', '1.1444091797e-05', '-1.66', '-0.00022125244141', '-1.67', '-0.00010681152344', '-1.68', '1.1444091797e-05', '-1.69', '-0.00010681152344', '-1.7', '1.1444091797e-05', '-1.71', '-0.00010681152344', '-1.72', '-0.00010681152344', '-1.73', '-0.00022125244141', '-1.74', '-0.00010681152344', '-1.75', '-0.00010681152344', '-1.76', '1.1444091797e-05', '-1.77', '1.1444091797e-05', '-1.78', '-0.00010681152344', '-1.79', '-0.00010681152344', '-1.8', '-0.00022125244141', '-1.81', '-0.00010681152344', '-1.82', '-0.00022125244141', '-1.83', '-0.00010681152344', '-1.84', '-0.00010681152344', '-1.85', '-0.00010681152344', '-1.86', '-0.00010681152344', '-1.87', '1.1444091797e-05', '-1.88', '1.1444091797e-05', '-1.89', '-0.00010681152344', '-1.9', '-0.00010681152344', '-1.91', '-0.00022125244141', '-1.92', '-0.00010681152344', '-1.93', '1.1444091797e-05', '-1.94', '1.1444091797e-05', '-1.95', '-0.00010681152344', '-1.96', '0.00012588500977', '-1.97', '-0.00022125244141', '-1.98', '1.1444091797e-05', '-1.99', '-0.00010681152344', '-2', '-0.00010681152344', '-2.01', '-0.00022125244141', '-2.02', '-0.00010681152344', '-2.03', '-0.00010681152344', '-2.04', '-0.00022125244141', '-2.05', '-0.00010681152344', '-2.06', '-0.00010681152344', '-2.07', '1.1444091797e-05', '-2.08', '1.1444091797e-05', '-2.09', '-0.00010681152344', '-2.1', '-0.00010681152344', '-2.11', '-0.00010681152344', '-2.12', '1.1444091797e-05', '-2.13', '1.1444091797e-05', '-2.14', '0.00012588500977', '-2.15', '-0.00010681152344', '-2.16', '-0.00010681152344', '-2.17', '-0.00010681152344', '-2.18', '1.1444091797e-05', '-2.19', '0.00012588500977', '-2.2', '-0.00010681152344', '-2.21', '-0.00022125244141', '-2.22', '-0.00010681152344', '-2.23', '1.1444091797e-05', '-2.24', '1.1444091797e-05', '-2.25', '-0.00010681152344', '-2.26', '-0.00022125244141', '-2.27', '-0.00010681152344', '-2.28', '-0.00010681152344', '-2.29', '-0.00010681152344', '-2.3', '-0.00010681152344', '-2.31', '-0.00022125244141', '-2.32', '1.1444091797e-05', '-2.33', '1.1444091797e-05', '-2.34', '-0.00010681152344', '-2.35', '1.1444091797e-05', '-2.36', '-0.00010681152344', '-2.37', '0.00012588500977', '-2.38', '-0.00010681152344', '-2.39', '-0.00022125244141', '-2.4', '-0.00010681152344', '-2.41', '-0.00010681152344', '-2.42', '1.1444091797e-05', '-2.43', '-0.00022125244141', '-2.44', '-0.00022125244141', '-2.45', '1.1444091797e-05', '-2.46', '1.1444091797e-05', '-2.47', '-0.00010681152344', '-2.48', '-0.00022125244141', '-2.49', '-0.00010681152344', '-2.5', '-0.00010681152344', '-2.51', '-0.00010681152344', '-2.52', '-0.00010681152344', '-2.53', '-0.00010681152344', '-2.54', '1.1444091797e-05', '-2.55', '-0.00010681152344', '-2.56', '1.1444091797e-05', '-2.57', '-0.00022125244141', '-2.58', '1.1444091797e-05', '-2.59', '-0.00010681152344', '-2.6', '-0.00010681152344', '-2.61', '1.1444091797e-05', '-2.62', '-0.00010681152344', '-2.63', '1.1444091797e-05', '-2.64', '-0.00010681152344', '-2.65', '1.1444091797e-05', '-2.66', '-0.00010681152344', '-2.67', '-0.00010681152344', '-2.68', '1.1444091797e-05', '-2.69', '-0.00010681152344', '-2.7', '0.00012588500977', '-2.71', '1.1444091797e-05', '-2.72', '1.1444091797e-05', '-2.73', '1.1444091797e-05', '-2.74', '-0.00010681152344', '-2.75', '-0.00010681152344', '-2.76', '-0.00010681152344', '-2.77', '-0.00010681152344', '-2.78', '1.1444091797e-05', '-2.79', '1.1444091797e-05', '-2.8', '1.1444091797e-05', '-2.81', '1.1444091797e-05', '-2.82', '-0.00022125244141', '-2.83', '1.1444091797e-05', '-2.84', '1.1444091797e-05', '-2.85', '1.1444091797e-05', '-2.86', '-0.00010681152344', '-2.87', '-0.00022125244141', '-2.88', '1.1444091797e-05', '-2.89', '1.1444091797e-05', '-2.9', '1.1444091797e-05', '-2.91', '1.1444091797e-05', '-2.92', '-0.00010681152344', '-2.93', '-0.00010681152344', '-2.94', '-0.00010681152344', '-2.95', '-0.00022125244141', '-2.96', '0.00012588500977', '-2.97', '-0.00022125244141', '-2.98', '1.1444091797e-05', '-2.99', '-0.00010681152344', '-3', '-0.00010681152344\n']
filePath = os.path.join(folderPath, 'defbuffer1_0818_130546.csv')
data = []
with open(filePath, newline = '') as file:
    rows = csv.reader(file)
    for i, row in enumerate(rows):
        if i > 8:
            data.append(row[13])
            data.append(row[0])
print(data)
Vstart = 0
Vstep = 0.05
Vstop = 2.85
stepNum = np.floor(abs((Vstart - Vstop)/Vstep)).astype(int) + 1
vRange = "AUTO"
iCompliance = 7
iNPLC = 0.01
delay = 0.0
sweepNum = 13
Sample = folderPath[-3:]
fileName = "Sweep"+str(sweepNum)+Sample+'Star'+str(Vstart).replace("-","M")+'_Stop'+str(Vstop).replace("-","M")+'_Step'+str(Vstep).replace("-", "M")+'.csv'
filePath = os.path.join(folderPath, fileName)
lstCurrent = [ float(i) for i in data[1::2]]# get all odd elements, which are currents

lstVoltage = [float(v) for v in data[0::2]] # get all even elements, which are voltages

# if Vstart > Vstop:
#     Vstep = -abs(Vstep)
# else:
#     Vstep = abs(Vstep)
lstAbsCurrent = [abs(i) for i in lstCurrent]
lstCalVoltage = [(Vstart + i*Vstep) for i in range(stepNum)]
dataToWrite = [['vRange', vRange]]
dataToWrite.append(['iCompliance', iCompliance])
#dataToWrite.append(['iRange', iRange[16:]])
dataToWrite.append(['iNPLC', iNPLC])
dataToWrite.append(['data Number', stepNum])
dataToWrite = [['V', 'I', 'V_cal','abs_I']]
for i in range(len(lstCurrent)):
    dataToWrite.append([lstVoltage[i], lstCurrent[i], lstCalVoltage[i] , lstAbsCurrent[i]])
try:
    with open(filePath, 'w', newline = '') as file:
        writer = csv.writer(file)
        writer.writerows(dataToWrite)
        file.close()
except IOError:
    print("Error: Could not open file for writing")

fig, ax = plt.subplots()
line, = ax.plot(lstVoltage, lstAbsCurrent, 'o-')
ax.set_xlabel('Voltage (V)')
ax.set_ylabel('Current (A)')
ax.set_yscale('linear')
plt.show()
plt.savefig(os.path.splitext(filePath)[0] + '_lin.png')
ax.set_yscale('log')
plt.show()
plt.savefig(os.path.splitext(filePath)[0] + '_log.png')

['0.0000079861675', '-2.92446691863e-11', '0.0500076971948', '-5.10488110117e-11', '0.1000067293644', '-3.326123404479e-11', '0.1500071138144', '-3.096605610819e-11', '0.2000108510256', '-1.317847914128e-11', '0.2500160634518', '-8.588123268094e-12', '0.3000130653381', '1.14946455132e-11', '0.3500243723392', '1.560901269348e-10', '0.4000240862370', '6.151215647954e-10', '0.4500201344490', '2.255010844721e-09', '0.5000258088112', '7.51378870234e-09', '0.5500237345695', '2.179310776285e-08', '0.6000254750252', '5.751089204864e-08', '0.6500219106674', '1.439934749214e-07', '0.7000170946121', '3.373288564035e-07', '0.7500163912773', '7.218386599561e-07', '0.8000247478485', '1.245172256858e-06', '0.8500169515610', '2.547401209085e-06', '0.9000145196915', '4.784446446138e-06', '0.9500044584274', '8.106871064228e-06', '1.0000202655792', '1.120285742218e-05', '1.0500179529190', '2.055442200799e-05', '1.1000063419342', '3.556591400411e-05', '1.1499996185303', '6.017425766913e-05', '1.2000106573

In [18]:
rm.close()

### REFERENCES

In [ ]:
iLimit = 7
#m2461.write(f"smu.source.ilimit = %f " % iLimit)
m2461.write(f"smu.source.ilimit.level = %f " % iLimit)

37

In [ ]:
vRange = "smu.ON"
m2461.write(f"smu.source.autorange = {vRange}") 

31

In [ ]:
m2461.write(f"beeper.beep(1,100)")
m2461.write(f"reset()")
m2461.write(f"status.reset()")
m2461.write(f"errorqueue.clear()")
# Configure source function as 2W DCVOLTS
m2461.write(f"smu.source.func = smu.FUNC_DC_VOLTAGE")
m2461.write(f"smu.measure.sense = smu.SENSE_2WIRE")
m2461.write(f"display.changescreen(display.SCREEN_HOME)")

43

In [ ]:
m2461.write(f'while defbuffer1.n < {stepNum} do delay(0.1) end')
data = m2461.query_ascii_values(f'printbuffer(1, {stepNum:f}, defbuffer1.sourcevalues, defbuffer1.readings)')
print(f'data: {data}')

m2461.write(f'print(1)')  # Send confirmation signal

In [ ]:
setV = 1
outputON = False
m2461.write(f"smu.measure.func = smu.FUNC_DC_CURRENT") 
m2461.write(f"smu.source.func = smu.FUNC_DC_VOLTAGE")
m2461.write(f"smu.source.level = %f" %setV)
if (not outputON):
    m2461.write(f"smu.source.output = 1")
m2461.write(f"print(smu.measure.read())") 
rawData = m2461.read()
# print("%g\t%g\t%g", smua.nvbuffer1.timestamps[1], smua.nvbuffer1.readings[1], smua.nvbuffer2.readings[1])
arrRawData = rawData.split(",")
print ("rawData: ", rawData)

rawData:  -7.4574048325e-11



In [ ]:
Vstart = 0
Vstop = 0.6
stepNum = 100
pulseWidth = 0.5e-3
delay = 0
offTime = 0.1
iCompliance = 7
m2461.write(f'reset()')
m2461.write(f'smu.source.func = smu.FUNC_DC_VOLTAGE ')
m2461.write(f'smu.source.range = 7 ')
m2461.write(f'smu.source.readback = smu.OFF')
m2461.write(f"smu.source.ilimit.level = %f" % iCompliance)
m2461.write(f'smu.measure.func = smu.FUNC_DC_CURRENT ')
m2461.write(f'smu.measure.range = 10 ')
m2461.write(f'smu.measure.nplc = 0.01 ')
m2461.write(f'smu.measure.autozero.enable = smu.OFF')





m2461.write(f'smu.source.pulsesweeplinear("VoltPulseLinSweepList", 0, {Vstart:0.2f}, {Vstop:0.2f}, {stepNum:f}, {pulseWidth:0.6f}, smu.ON, defbuffer1, {delay:0.6f}, {offTime:0.6f}, 1, 7, 10, smu.OFF, smu.OFF)') 


m2461.write(f'trigger.model.initiate()')

 




26